In [28]:
import json
from pathlib import Path
from itertools import product
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.append(os.path.abspath("../"))
from simulators.drossel_schwabl_CA import DrosselSchwablForestFire
from simulators.CA_modified import CellularAutomaton_humidity_age
from simulators.metrics import FireMetrics

In [29]:
DATASET_DIR = Path("synthetic_fire_dataset")

env = np.load(DATASET_DIR / "common_maps.npz")
height_grid = env["height_grid"]
age_grid = env["age_grid"]
moisture_grid = env["moisture_grid"]

with open(DATASET_DIR / "dataset_index.json", "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

print("Nombre de feux :", len(dataset_index))
print("Shape :", height_grid.shape)

N_STEPS = 50

Nombre de feux : 20
Shape : (100, 100)


In [30]:
def make_phi(gamma):
    def phi(delta_h):
        if delta_h <= 0:
            return np.exp(gamma * delta_h)
        else:
            return 1.0 + gamma * np.sqrt(delta_h)
    return phi

def make_psi(beta):
    def psi(m):
        return np.exp(-beta * m)
    return psi

In [31]:
def initialize_center_pixel(ca, value=1.0):
    H, W = ca.get_state().shape
    ca.initialize_ignition([(H // 2, W // 2)], [value])

In [32]:
def simulate_fire_candidate(
    height_grid,
    age_grid,
    moisture_grid,
    wind_grid,
    alpha,
    beta,
    gamma,
    n_steps=N_STEPS,
):
    H, W = height_grid.shape

    ca = CellularAutomaton_humidity_age(
        grid_height=H,
        grid_width=W,
        wind_grid=wind_grid,
        height_grid=height_grid,
        age_grid=age_grid,
        moisture_grid=moisture_grid,
        phi=make_phi(gamma),
        psi=make_psi(beta),
        alpha_age=alpha,
    )

    initialize_center_pixel(ca, value=0.8)

    for _ in range(n_steps):
        ca.evolve(use_age=True, use_moisture=True)

    return ca.get_state().copy().astype(np.float32)

In [33]:
def iou_score(pred, obs, tau=0.5):
    pred_bin = pred >= tau
    obs_bin = obs >= tau

    inter = np.logical_and(pred_bin, obs_bin).sum()
    union = np.logical_or(pred_bin, obs_bin).sum()

    if union == 0:
        return 1.0
    return inter / union

In [34]:
def evaluate_one_fire_iou(
    fire_path,
    height_grid,
    age_grid,
    moisture_grid,
    alpha,
    beta,
    gamma,
    n_steps=N_STEPS,
    tau=0.5,
):
    obs = np.load(fire_path)

    wind_grid = obs["wind_grid"].astype(np.float32)
    obs_final = obs["final_state"].astype(np.float32) / 255.0

    pred_final = simulate_fire_candidate(
        height_grid=height_grid,
        age_grid=age_grid,
        moisture_grid=moisture_grid,
        wind_grid=wind_grid,
        alpha=alpha,
        beta=beta,
        gamma=gamma,
        n_steps=n_steps,
    )

    iou = iou_score(pred_final, obs_final, tau=tau)

    return {
        "iou": float(iou),
        "iou_loss": float(1.0 - iou),
    }

In [35]:
def evaluate_params_iou(
    fire_paths,
    height_grid,
    age_grid,
    moisture_grid,
    alpha,
    beta,
    gamma,
    n_steps=N_STEPS,
    tau=0.5,
):
    scores = []

    for fire_path in fire_paths:
        s = evaluate_one_fire_iou(
            fire_path=fire_path,
            height_grid=height_grid,
            age_grid=age_grid,
            moisture_grid=moisture_grid,
            alpha=alpha,
            beta=beta,
            gamma=gamma,
            n_steps=n_steps,
            tau=tau,
        )
        scores.append(s["iou"])

    mean_iou = float(np.mean(scores))

    return {
        "alpha": alpha,
        "beta": beta,
        "gamma": gamma,
        "mean_iou": mean_iou,
        "mean_iou_loss": 1.0 - mean_iou,
    }

In [38]:
alpha_grid = [0.8, 2.0, 4.0]
beta_grid  = [0.5, 2.0, 5.0]
gamma_grid = [0.4, 1.2, 3.0]

N_EVAL_FIRES = 3

fire_paths = [
    DATASET_DIR / item["file"]
    for item in dataset_index[:N_EVAL_FIRES]
]

print("Nombre de triplets testés :", len(alpha_grid) * len(beta_grid) * len(gamma_grid))
print("Nombre de feux utilisés :", len(fire_paths))

Nombre de triplets testés : 27
Nombre de feux utilisés : 3


In [39]:
results = []

for alpha, beta, gamma in product(alpha_grid, beta_grid, gamma_grid):
    print(f"Test alpha={alpha}, beta={beta}, gamma={gamma}")

    row = evaluate_params_iou(
        fire_paths=fire_paths,
        height_grid=height_grid,
        age_grid=age_grid,
        moisture_grid=moisture_grid,
        alpha=alpha,
        beta=beta,
        gamma=gamma,
        n_steps=N_STEPS,
        tau=0.5,
    )

    results.append(row)

results_df = pd.DataFrame(results)

Test alpha=0.8, beta=0.5, gamma=0.4
Test alpha=0.8, beta=0.5, gamma=1.2
Test alpha=0.8, beta=0.5, gamma=3.0
Test alpha=0.8, beta=2.0, gamma=0.4
Test alpha=0.8, beta=2.0, gamma=1.2
Test alpha=0.8, beta=2.0, gamma=3.0
Test alpha=0.8, beta=5.0, gamma=0.4
Test alpha=0.8, beta=5.0, gamma=1.2
Test alpha=0.8, beta=5.0, gamma=3.0
Test alpha=2.0, beta=0.5, gamma=0.4
Test alpha=2.0, beta=0.5, gamma=1.2
Test alpha=2.0, beta=0.5, gamma=3.0
Test alpha=2.0, beta=2.0, gamma=0.4
Test alpha=2.0, beta=2.0, gamma=1.2
Test alpha=2.0, beta=2.0, gamma=3.0
Test alpha=2.0, beta=5.0, gamma=0.4
Test alpha=2.0, beta=5.0, gamma=1.2
Test alpha=2.0, beta=5.0, gamma=3.0
Test alpha=4.0, beta=0.5, gamma=0.4
Test alpha=4.0, beta=0.5, gamma=1.2
Test alpha=4.0, beta=0.5, gamma=3.0
Test alpha=4.0, beta=2.0, gamma=0.4
Test alpha=4.0, beta=2.0, gamma=1.2
Test alpha=4.0, beta=2.0, gamma=3.0
Test alpha=4.0, beta=5.0, gamma=0.4
Test alpha=4.0, beta=5.0, gamma=1.2
Test alpha=4.0, beta=5.0, gamma=3.0


In [40]:
df = results_df.copy()
results_df.sort_values("mean_iou_loss").head(10)

best = results_df.sort_values("mean_iou_loss").iloc[0]

print("Meilleurs paramètres trouvés :")
print(best[["alpha", "beta", "gamma", "mean_iou", "mean_iou_loss"]])

Meilleurs paramètres trouvés :
alpha            0.800000
beta             5.000000
gamma            0.400000
mean_iou         0.536338
mean_iou_loss    0.463662
Name: 6, dtype: float64


In [43]:
print(results_df.sort_values("mean_iou_loss"))

    alpha  beta  gamma  mean_iou  mean_iou_loss
6     0.8   5.0    0.4  0.536338       0.463662
15    2.0   5.0    0.4  0.535426       0.464574
16    2.0   5.0    1.2  0.523167       0.476833
7     0.8   5.0    1.2  0.522026       0.477974
24    4.0   5.0    0.4  0.515941       0.484059
25    4.0   5.0    1.2  0.504787       0.495213
17    2.0   5.0    3.0  0.499149       0.500851
8     0.8   5.0    3.0  0.497283       0.502717
26    4.0   5.0    3.0  0.484153       0.515847
21    4.0   2.0    0.4  0.190553       0.809447
22    4.0   2.0    1.2  0.186956       0.813044
23    4.0   2.0    3.0  0.180529       0.819471
12    2.0   2.0    0.4  0.174087       0.825913
13    2.0   2.0    1.2  0.170919       0.829081
14    2.0   2.0    3.0  0.165288       0.834712
3     0.8   2.0    0.4  0.159069       0.840931
4     0.8   2.0    1.2  0.156078       0.843922
5     0.8   2.0    3.0  0.150541       0.849459
18    4.0   0.5    0.4  0.135493       0.864507
19    4.0   0.5    1.2  0.134143       0

In [45]:
true_alpha = 2.0
true_beta = 2.0
true_gamma = 1.2

fire_path = fire_paths[0]
score = evaluate_one_fire_iou(
    fire_path=fire_path,
    height_grid=height_grid,
    age_grid=age_grid,
    moisture_grid=moisture_grid,
    alpha=true_alpha,
    beta=true_beta,
    gamma=true_gamma,
    n_steps=N_STEPS,   # exactement le même que génération
    tau=0.5,
)

print(score)

{'iou': 0.17452371019329718, 'iou_loss': 0.8254762898067028}
